# Pre-TEST3 — Qwen forced-fallback cache (80 Tables)

**Runtime:** Colab GPU (T4) + Linux.

This notebook runs the high-value experiment:
1. `generate_cache()` — Stage3 prompts, rules never consulted
2. `replay_cache()` — same codes on Stage3 / TEST1 / TEST2 executors
3. Download `generations_*.json` + replay CSV for local H1–H6 analysis

Not a private submission. Not the 40s solve benchmark.

## 0. Mount / upload project

Upload a zip of `projects/` (must include `audit_independent/`, `CODEX_AUDIT_HANDOFF/`, `src/`) or clone your private mirror.
Set `PROJECT_ROOT` below.

In [ ]:
import os, sys, platform, subprocess
from pathlib import Path

# --- EDIT THIS ---
# Clone: !git clone https://github.com/BeediGoua/TEST-PROJET-ML.git
# Ce repo = contenu de `projects/` → PROJECT_ROOT = racine du clone
PROJECT_ROOT = Path('/content/TEST-PROJET-ML')
# -----------------

assert PROJECT_ROOT.exists(), f'missing {PROJECT_ROOT}'
AUDIT = PROJECT_ROOT / 'audit_independent'
sys.path.insert(0, str(AUDIT))
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(AUDIT)
print('cwd', Path.cwd())
print('platform', platform.platform())

import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'Enable GPU runtime (Runtime → Change runtime type → T4)'
assert platform.system() == 'Linux'

In [ ]:
# Dependencies (match local audit / official harness)
%pip -q install 'torch' 'transformers>=4.40' 'accelerate' 'pandas' 'numpy' 'scipy' 'scikit-learn' 'sympy' 'mlarena'

In [ ]:
from gpu_experiments import generate_cache, replay_cache, summarize_replay

cache_path = generate_cache(
    model='Qwen/Qwen2.5-Coder-1.5B-Instruct',
    prompt_agent='stage3',
    seed=0,
)
print('CACHE', cache_path)

In [ ]:
rows = replay_cache(cache_path, timeout=15.0)
summary = summarize_replay(rows)
display(summary)
summary.to_csv(str(cache_path).replace('.json', '_summary.csv'))
print('replay rows', len(rows))

In [ ]:
from google.colab import files
from pathlib import Path
p = Path(cache_path)
for f in [p, Path(str(p).replace('.json', '_replay.csv')), Path(str(p).replace('.json', '_summary.csv'))]:
    if f.exists():
        print('download', f)
        files.download(str(f))
    else:
        print('missing', f)
print('Place generations_*.json under projects/audit_independent/results/ then run diagnostic/07–12 locally.')